# 17.14 模型能預先適應量化誤差嗎？

# 第 17 章：量化，用更少位元表示模型

前置：第2章Linear與數值。像把連續刻度尺改成較粗的格子：先看誤差，再看真正的儲存bytes，最後才談硬體加速。quantization.py的參考Linear會反量化，不會假裝是低位元kernel。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：換較粗刻度尺：省格子但有誤差**

縮小儲存與硬體加速分開驗證。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/quantization.svg")))

**先想一想：**forward用了round，爲什麼grad還不是0？

QAT讓forward先體驗刻度誤差，再用近似梯度學習。round本身幾乎處處導數0，straight-through estimator把backward近似成恆等。

**把直覺寫成數學：**fake_quant=x+(dequant(quant(x))−x).detach()；forward與backward使用不同規則。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.quantization import fake_quantize

x = torch.tensor([0.1, 0.4, 0.9], requires_grad=True)
y = fake_quantize(x, 4)
y.sum().backward()
print("forward", y.detach(), "近似grad", x.grad)
assert torch.equal(x.grad, torch.ones_like(x))

**如何讀結果：**這裏只檢查STE；真的QAT需訓練後convert再評，不把fake quant當已壓縮權重文件。

**只改一件事：**只移除detach構造，觀察round梯度造成的差異。
